# Phase 1 — Data Cleaning

This notebook walks through the full ingestion and cleaning pipeline for the Ames Housing dataset. The dataset contains 2,919 residential home sales in Ames, Iowa from 2006 to 2010, with 79 explanatory variables.

**Source:** Dean De Cock (2011). *Ames, Iowa: Alternative to the Boston Housing Data Set.* Journal of Statistics Education, 19(3). Fetched via OpenML dataset ID 42165.

**Pipeline:**
1. Fetch from OpenML and load into SQLite (`raw_housing` table)
2. Diagnose missing values
3. Apply cleaning functions from `src/clean.py`
4. Compare before/after
5. Write cleaned data to `clean_housing` table and `data/processed/`

In [ ]:
import sys
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import ingest
import clean

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

DB_PATH = Path.cwd().parent / 'data' / 'housing.db'

## 1. Ingest raw data

In [ ]:
# Fetch from OpenML and load into SQLite. Skips download if CSV already exists.
raw_csv = Path.cwd().parent / 'data' / 'raw' / 'ames_housing.csv'
if raw_csv.exists():
    print(f'Raw CSV already exists at {raw_csv}, loading from disk.')
    df_raw = pd.read_csv(raw_csv)
    conn = sqlite3.connect(DB_PATH)
    schema_sql = (Path.cwd().parent / 'sql' / 'schema.sql').read_text()
    conn.executescript(schema_sql)
    conn.execute('DELETE FROM raw_housing')
    df_raw.to_sql('raw_housing', conn, if_exists='append', index=False)
    conn.commit()
    conn.close()
else:
    df_raw = ingest.run()

print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head(3)

## 2. Missing value audit

In [ ]:
missing = (
    df_raw.isnull().sum()
    .rename('null_count')
    .to_frame()
    .assign(pct=lambda d: (d['null_count'] / len(df_raw) * 100).round(1))
    .query('null_count > 0')
    .sort_values('null_count', ascending=False)
)
print(f'{len(missing)} columns have missing values')
missing

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cols_to_plot = missing[missing['null_count'] > 0].head(20)
ax.barh(cols_to_plot.index, cols_to_plot['pct'], color='steelblue')
ax.set_xlabel('% Missing')
ax.set_title('Missing values by column (top 20)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

Most high-missingness columns (`pool_qc`, `misc_feature`, `alley`, `fence`) represent the *absence* of a feature rather than truly missing data. The dataset author encodes these as `NA` strings in the original files. The cleaning pipeline converts them to `'None'` to distinguish them from null.

`lot_frontage` (18% missing) is genuinely missing and is imputed by neighborhood median.

## 3. Outlier check: above-grade living area

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sp = pd.to_numeric(df_raw['sale_price'], errors='coerce')
gla = pd.to_numeric(df_raw['gr_liv_area'], errors='coerce')
ax.scatter(gla, sp, alpha=0.3, s=10, color='steelblue')
ax.scatter(
    gla[(gla > 4000) & (sp < 200000)],
    sp[(gla > 4000) & (sp < 200000)],
    color='tomato', s=40, label='Flagged outliers'
)
ax.set_xlabel('Above-grade living area (sq ft)')
ax.set_ylabel('Sale price ($)')
ax.set_title('Sale price vs. living area — outlier identification')
ax.legend()
plt.tight_layout()
plt.show()

Two properties (red) have over 4,000 sq ft of living area but sold for well under $200k. The dataset author notes these are partial sales that do not reflect market value. They are removed before modeling.

## 4. Run cleaning pipeline

In [ ]:
df_clean = clean.run()
print(f'Clean dataset shape: {df_clean.shape}')
df_clean.head(3)

## 5. Before/after comparison

In [ ]:
print('Row counts')
print(f'  Raw:   {len(df_raw):,}')
print(f'  Clean: {len(df_clean):,}')
print(f'  Dropped: {len(df_raw) - len(df_clean)} (2 outliers)')

# Null counts after cleaning
remaining_nulls = df_clean.isnull().sum().sum()
print(f'\nTotal null values remaining in clean data: {remaining_nulls}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Sale price distribution
axes[0].hist(df_clean['sale_price'], bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Sale price ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sale price distribution')

# Living area box before vs after outlier removal
raw_gla = pd.to_numeric(df_raw['gr_liv_area'], errors='coerce').dropna()
axes[1].boxplot(
    [raw_gla, df_clean['gr_liv_area']],
    labels=['Raw', 'Clean'],
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6)
)
axes[1].set_ylabel('Above-grade living area (sq ft)')
axes[1].set_title('Living area: raw vs. clean')

plt.tight_layout()
plt.show()

The sale price distribution is right-skewed (expected for housing data). The living area boxplot shows the two extreme outliers are removed in the clean dataset, bringing the upper tail in line with the bulk of the data.

The cleaned dataset has **no missing values** and is ready for exploratory analysis.

In [ ]:
df_clean.describe(include='all').T[['count', 'mean', 'std', 'min', '50%', 'max']].head(20)